In [3]:
import os 
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

if os.path.exists(os.path.join("Data", "Raw")):
  raw_path = os.path.join("Data", "Raw")
elif os.path.exists(os.path.join("..", "Data", "Raw")):
  raw_path = os.path.join("..", "Data", "Raw")
else:
  raw_path = os.path.join("..", "..", "Data", "Raw")

print("Ham veriler okunuyor...\n")

df_orders = pd.read_csv(os.path.join(raw_path, "olist_orders_dataset.csv"))
df_items = pd.read_csv(os.path.join(raw_path, "olist_order_items_dataset.csv"))
df_products = pd.read_csv(os.path.join(raw_path, "olist_products_dataset.csv"))
df_customers = pd.read_csv(
    os.path.join(raw_path, "olist_customers_dataset.csv")
)
df_payments = pd.read_csv(
    os.path.join(raw_path, "olist_order_payments_dataset.csv")
)
df_translation = pd.read_csv(
    os.path.join(raw_path, "product_category_name_translation.csv")
)

print(f"✔ Siparişler (Orders): {df_orders.shape[0]:,} satır")
print(f"✔ Sipariş Detayları (Order Items): {df_items.shape[0]:,} satır")
print(f"✔ Ürünler (Products): {df_products.shape[0]:,} satır")
print(f"✔ Müşteriler (Customers): {df_customers.shape[0]:,} satır")
print(f"✔ Ödemeler (Payments): {df_payments.shape[0]:,} satır")









Ham veriler okunuyor...

✔ Siparişler (Orders): 99,441 satır
✔ Sipariş Detayları (Order Items): 112,650 satır
✔ Ürünler (Products): 32,951 satır
✔ Müşteriler (Customers): 99,441 satır
✔ Ödemeler (Payments): 103,886 satır


In [4]:
# 1. Kategori isimlerini anlaşılır İngilizce/Türkçe isimlerle birleştirelim
df_products = df_products.merge(
    df_translation, on="product_category_name", how="left"
)
df_products["category_name"] = df_products[
    "product_category_name_english"
].fillna("Diğer")

# 2. Tarih sütunlarını datetime formatına çevirelim
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
  df_orders[col] = pd.to_datetime(df_orders[col])

# 3. Siparişler, Kalemler, Ürünler ve Müşterileri Master Tabloda birleştirelim
df_master = df_orders.merge(df_items, on="order_id", how="inner")
df_master = df_master.merge(df_products, on="product_id", how="left")
df_master = df_master.merge(df_customers, on="customer_id", how="left")

# 4. Ödeme bilgilerini ekleyelim
df_payments_agg = (
    df_payments.groupby("order_id")
    .agg({"payment_value": "sum", "payment_type": "first"})
    .reset_index()
)
df_master = df_master.merge(df_payments_agg, on="order_id", how="left")

# 5. Toplam Harcama Tutarı
df_master["total_price"] = df_master["price"] + df_master["freight_value"]

print(
    "✔ Master Veri Seti Başarıyla Oluşturuldu! Toplam Satır Sayısı:"
    f" {len(df_master):,}\n"
)
df_master.head(3)

✔ Master Veri Seti Başarıyla Oluşturuldu! Toplam Satır Sayısı: 112,650



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,category_name,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_type,total_price
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.00,268.00,4.00,500.00,19.00,8.00,13.00,housewares,housewares,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,credit_card,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.00,178.00,1.00,400.00,19.00,13.00,19.00,perfumery,perfumery,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,boleto,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.00,232.00,1.00,420.00,24.00,19.00,21.00,auto,auto,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,credit_card,179.12


In [5]:
# Dinamik hedef klasör tespiti
if os.path.exists("Data"):
  clean_path = os.path.join("Data", "Clean")
elif os.path.exists(os.path.join("..", "Data")):
  clean_path = os.path.join("..", "Data", "Clean")
else:
  clean_path = os.path.join("..", "..", "Data", "Clean")

os.makedirs(clean_path, exist_ok=True)

# Kayıt işlemleri
df_master.to_csv(
    os.path.join(clean_path, "master_orders_clean.csv"), index=False
)
df_customers.to_csv(
    os.path.join(clean_path, "customers_clean.csv"), index=False
)
df_products.to_csv(os.path.join(clean_path, "products_clean.csv"), index=False)

print(
    "🎉 İŞLEM TAMAMLANDI! Temizlenmiş veriler 'Data/Clean/' klasörüne"
    " kaydedildi."
)

🎉 İŞLEM TAMAMLANDI! Temizlenmiş veriler 'Data/Clean/' klasörüne kaydedildi.
